# Bangladesh Road Accidents 2021–2026: Data Cleaning

This notebook builds a clean dataset of road accidents in Bangladesh from **January 2021 to September 2026**.

**Source.** All figures come from the **Road Safety Foundation (RSF)**, a Dhaka-based NGO that counts road
accidents reported by national dailies, online portals and TV every month and publishes monthly and annual reports.
RSF does not release a downloadable dataset, so the numbers are transcribed from the press articles that reproduce
its reports:

1. `data/raw/deadly_roads/`: CSVs from the open
   [Deadly Roads: Bangladesh 2000–2025](https://github.com/ahasan722/bangladesh-road-crashes-2000-2025)
   project (CC BY 4.0), with one press citation per row.
2. `data/raw/rsf_manual_additions.csv`: extra RSF figures, checked against the original articles.

A second, official source is added in section 8: the **Bangladesh Road Transport Authority (BRTA)** monthly reports
(Jan 2023 – Sep 2026), which give all 8 divisions and 13 vehicle types every month.

Every output table keeps `source` and `source_url` columns, so each number can be traced to the article or report
it came from.

**Outputs** (written to `data/processed/`):

| File | One row per |
|---|---|
| `monthly_national.csv` | month (Jan 2021 – Sep 2026, including months with no report) |
| `annual_national.csv` | year |
| `vehicle_deaths.csv` | period × vehicle type |
| `division.csv` | period × division × measure |
| `time_of_day.csv` | period × time band |
| `eid_periods.csv` | Eid travel window |
| `brta_monthly_national.csv` | month (BRTA, Jan 2023 – Sep 2026) |
| `brta_division.csv` | month × division (BRTA) |
| `brta_vehicle_deaths.csv` | month × vehicle type (BRTA) |

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

RAW = Path("../data/raw")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

START, END = pd.Period("2021-01", "M"), pd.Period("2026-09", "M")

## 1. Load the raw files

In [2]:
monthly_raw = pd.read_csv(RAW / "deadly_roads/monthly.csv")
breakdowns_raw = pd.read_csv(RAW / "deadly_roads/breakdowns.csv")
manual = pd.read_csv(RAW / "rsf_manual_additions.csv")

for name, df in [("monthly", monthly_raw), ("breakdowns", breakdowns_raw), ("manual", manual)]:
    print(f"{name:<11} {df.shape[0]:>5} rows  {df.shape[1]:>2} cols")

monthly_raw.head()

monthly        58 rows   8 cols
breakdowns   2471 rows   9 cols
manual         30 rows  10 cols


,source,year,month,accidents,deaths,injuries,citation_url,note
0,RSF,2020,1,340.0,445,NaN,https://www.dhakatribune.com/bangladesh/237902...,back-reference in Jan 2021 article
1,RSF,2020,8,302.0,379,NaN,https://en.prothomalo.com/bangladesh/road-acci...,NaN
2,RSF,2021,1,427.0,484,673.0,https://www.dhakatribune.com/bangladesh/237902...,NaN
3,RSF,2021,5,441.0,562,NaN,https://mail.thefinancialexpress.com.bd/nation...,back-reference
4,RSF,2021,6,327.0,398,423.0,https://mail.thefinancialexpress.com.bd/nation...,NaN


## 2. Data quality checks

Issues to look for before using the data:

- **Several organisations.** The monthly file mixes RSF with Bangladesh Jatri Kalyan Samity (BJKS) and SCRF. Their
  methods differ, so the series must not be mixed.
- **Non-numeric years**, such as `unknown` or ranges like `2020-2026`.
- **Wrong years.** Monthly press articles often say "in June" without a year, so a row can be filed under the
  wrong year.
- **Duplicates.** The same figure can be cited from more than one article.

In [3]:
print("Sources in monthly file:", monthly_raw["source"].value_counts().to_dict())
print("Non-numeric years (monthly):", monthly_raw.loc[pd.to_numeric(monthly_raw["year"], errors="coerce").isna(), "year"].unique().tolist())
print("Non-numeric years (breakdowns):", breakdowns_raw.loc[pd.to_numeric(breakdowns_raw["year"], errors="coerce").isna(), "year"].unique().tolist())
print("Missing deaths (monthly):", monthly_raw["deaths"].isna().sum())

Sources in monthly file: {'RSF': 45, 'BJKS': 7, 'SCRF': 6}
Non-numeric years (monthly): ['unknown']
Non-numeric years (breakdowns): ['2020-2026', '2019-2024', 'unknown', '2014-2024']
Missing deaths (monthly): 0


### 2.1 Fix wrong or unknown years

I checked the publication date of every cited article for the monthly RSF rows. Two need fixing:

| Article | Filed as | Published | Correct month |
|---|---|---|---|
| TBS node 646738 (408 deaths) | `unknown`-05 | 2023-06-09 | **May 2023** |
| Daily Star "696 killed in road accidents in June" | 2024-06 | 2025-07-02 | **June 2025** |

The fix is keyed on the citation URL, so the same correction also applies to every breakdown row taken from that
article.

In [4]:
YEAR_FIXES = {
    "https://www.tbsnews.net/node/646738": (2023, 5),
    "https://tds-images.thedailystar.net/news/bangladesh/accidents-fires/news/696-killed-road-accidents-june-3930696": (2025, 6),
}


def fix_years(df):
    df = df.copy()
    for url, (year, month) in YEAR_FIXES.items():
        hit = df["citation_url"] == url
        df.loc[hit, "year"] = str(year)  # the raw year column is text (it contains 'unknown')
        if "month" in df:
            df.loc[hit, "month"] = month
        if "period" in df:
            df.loc[hit, "period"] = f"{year}-{month:02d}"
    return df


monthly_fixed = fix_years(monthly_raw)
breakdowns = fix_years(breakdowns_raw)
monthly_fixed.loc[monthly_fixed["citation_url"].isin(YEAR_FIXES), ["source", "year", "month", "accidents", "deaths", "injuries"]]

,source,year,month,accidents,deaths,injuries
24,RSF,2025,6,689.0,696,1867.0
31,RSF,2023,5,491.0,408,631.0


## 3. Monthly national totals

Keep RSF only, add the manually verified months, then reindex onto the full calendar so that months with no
report appear as `NaN` instead of being silently missing.

In [5]:
rsf_monthly = (
    monthly_fixed[monthly_fixed["source"] == "RSF"]
    .rename(columns={"citation_url": "source_url"})
    .assign(year=lambda d: d["year"].astype(int), month=lambda d: d["month"].astype(int))
    [["year", "month", "accidents", "deaths", "injuries", "source", "source_url", "note"]]
)

manual_monthly = manual[manual["table"] == "monthly"][
    ["year", "month", "accidents", "deaths", "injuries", "source", "source_url", "note"]
]

monthly = pd.concat([rsf_monthly, manual_monthly], ignore_index=True)
monthly["period"] = pd.PeriodIndex.from_fields(year=monthly["year"], month=monthly["month"], freq="M")
monthly = monthly[(monthly["period"] >= START) & (monthly["period"] <= END)]

dupes = monthly[monthly.duplicated("period", keep=False)]
print("Duplicate months:", len(dupes))
monthly = monthly.drop_duplicates("period", keep="first")

calendar = pd.DataFrame({"period": pd.period_range(START, END, freq="M")})
monthly = calendar.merge(monthly, on="period", how="left")
monthly["year"] = monthly["period"].dt.year
monthly["month"] = monthly["period"].dt.month
monthly["month_name"] = monthly["period"].dt.strftime("%b")
monthly["reported"] = monthly["deaths"].notna()
monthly["source"] = monthly["source"].fillna("RSF")

print(f"Months in range: {len(monthly)}, with RSF data: {monthly['reported'].sum()}")

Duplicate months: 0
Months in range: 69, with RSF data: 51


### 3.1 Coverage

Which months have an RSF figure (✓) and which have none (·):

In [6]:
coverage = monthly.pivot(index="year", columns="month", values="reported").replace({True: "✓", False: "·"})
coverage.columns = [pd.Timestamp(2000, m, 1).strftime("%b") for m in coverage.columns]
coverage.fillna("")

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
year,,,,,,,,,,,,
2021,✓,·,✓,✓,✓,✓,·,·,·,✓,✓,✓
2022,·,·,✓,✓,✓,✓,✓,✓,✓,·,✓,·
2023,·,✓,✓,·,✓,✓,✓,✓,✓,✓,✓,✓
2024,·,✓,✓,✓,·,✓,✓,·,✓,✓,·,·
2025,✓,✓,✓,✓,·,✓,✓,✓,✓,✓,✓,✓
2026,✓,·,✓,✓,·,✓,✓,✓,✓,,,


### 3.2 Sanity checks

- Deaths per accident should sit around 0.8–1.4. Values far outside that suggest a transcription error.
- Injuries per accident vary much more, because RSF's injury counts depend on how many injured people the press
  reports.

In [7]:
monthly["deaths_per_accident"] = (monthly["deaths"] / monthly["accidents"]).round(2)
monthly["injuries_per_accident"] = (monthly["injuries"] / monthly["accidents"]).round(2)
monthly["deaths_per_day"] = (monthly["deaths"] / monthly["period"].dt.days_in_month).round(2)

print(monthly["deaths_per_accident"].describe().round(2).to_string())
monthly.loc[
    (monthly["deaths_per_accident"] < 0.8) | (monthly["deaths_per_accident"] > 1.4) | (monthly["injuries_per_accident"] > 3),
    ["period", "accidents", "deaths", "injuries", "deaths_per_accident", "injuries_per_accident"],
]

count    49.00
mean      1.04
std       0.13
min       0.83
25%       0.93
50%       0.99
75%       1.13
max       1.29


,period,accidents,deaths,injuries,deaths_per_accident,injuries_per_accident
18,2022-07,632.0,739.0,2042.0,1.17,3.23
62,2026-03,576.0,532.0,2221.0,0.92,3.86


Mar 2026 (2,221 injured from 576 accidents) is the only extreme value. It matches the source article, so I keep it
but flag it. July 2022 (2,042 injured) is the Eid-ul-Azha month and is plausible.

In [8]:
monthly.loc[monthly["period"] == pd.Period("2026-03", "M"), "note"] = "injury count as reported; unusually high"

monthly_cols = [
    "year", "month", "month_name", "accidents", "deaths", "injuries", "deaths_per_day",
    "deaths_per_accident", "reported", "source", "source_url", "note",
]
monthly[monthly_cols].to_csv(OUT / "monthly_national.csv", index=False)
monthly[monthly_cols].head()

,year,month,month_name,accidents,deaths,injuries,deaths_per_day,deaths_per_accident,reported,source,source_url,note
0,2021,1,Jan,427.0,484.0,673.0,15.61,1.13,True,RSF,https://www.dhakatribune.com/bangladesh/237902...,NaN
1,2021,2,Feb,NaN,NaN,NaN,NaN,NaN,False,RSF,NaN,NaN
2,2021,3,Mar,409.0,513.0,598.0,16.55,1.25,True,RSF,https://old.thefinancialexpress.com.bd/nationa...,published 2021-04-05
3,2021,4,Apr,397.0,452.0,NaN,15.07,1.14,True,RSF,https://thefinancialexpress.com.bd/national/56...,back-reference in May 2021 report
4,2021,5,May,441.0,562.0,NaN,18.13,1.27,True,RSF,https://mail.thefinancialexpress.com.bd/nation...,back-reference


## 4. Annual national totals

RSF's annual reports (2021–2025) are complete counts for the year. For 2026 only January–September are in range,
and two of those months (Feb, May) were never reported, so the 2026 row is a **partial** sum and must not be
compared directly with full years.

In [9]:
annual_rows = breakdowns[(breakdowns["source"] == "RSF") & (breakdowns["period"] == "annual")].copy()
annual_rows["year"] = pd.to_numeric(annual_rows["year"], errors="coerce")
annual_rows = annual_rows[annual_rows["year"].between(2021, 2025)]

# Several articles report the same annual figure; keep the first citation (the report-launch article).
annual_rows = annual_rows.drop_duplicates(["year", "dimension", "category", "measure"], keep="first")

PICK = {
    ("total", "all", "accidents"): "accidents",
    ("total", "all", "deaths"): "deaths",
    ("total", "all", "injuries"): "injuries",
    ("vehicle_type", "motorcycle", "accidents"): "motorcycle_accidents",
    ("vehicle_type", "motorcycle", "deaths"): "motorcycle_deaths",
    ("victim", "pedestrian", "deaths"): "pedestrian_deaths",
    ("victim", "women", "deaths"): "women_deaths",
    ("victim", "children", "deaths"): "children_deaths",
}
annual_rows["field"] = [PICK.get(k) for k in zip(annual_rows["dimension"], annual_rows["category"], annual_rows["measure"])]
annual_rows = annual_rows.dropna(subset=["field"])

annual = annual_rows.pivot(index="year", columns="field", values="value")[list(PICK.values())]
annual["source_url"] = (
    annual_rows[annual_rows["field"] == "deaths"].set_index("year")["citation_url"]
)
annual["coverage"] = "full year"
annual.index = annual.index.astype(int)

# Fill gaps from the manually verified annual figures.
MANUAL_FIELDS = {("pedestrian", "deaths"): "pedestrian_deaths",
                 ("motorcycle", "accidents"): "motorcycle_accidents",
                 ("motorcycle", "deaths"): "motorcycle_deaths"}
for _, r in manual[manual["table"] == "annual"].iterrows():
    for (category, column), field in MANUAL_FIELDS.items():
        if r["category"] == category and pd.notna(r[column]) and pd.isna(annual.loc[int(r["year"]), field]):
            annual.loc[int(r["year"]), field] = r[column]
annual

field,accidents,deaths,injuries,motorcycle_accidents,motorcycle_deaths,pedestrian_deaths,women_deaths,children_deaths,source_url,coverage
year,,,,,,,,,,
2021,5371.0,6284.0,7468.0,2078.0,2214.0,1523.0,927.0,734.0,https://unb.com.bd/category/Bangladesh/perilou...,full year
2022,6829.0,7713.0,12615.0,2973.0,3091.0,1627.0,1061.0,1143.0,https://old.thefinancialexpress.com.bd/nationa...,full year
2023,6911.0,6524.0,11407.0,2532.0,2487.0,NaN,974.0,1128.0,https://www.dhakatribune.com/bangladesh/337884...,full year
2024,6927.0,7294.0,12019.0,2761.0,2609.0,1535.0,893.0,1152.0,https://en.prothomalo.com/bangladesh/m4kzejfpvx,full year
2025,7584.0,7359.0,16476.0,3029.0,2671.0,1564.0,962.0,1008.0,https://en.prothomalo.com/bangladesh/accident/...,full year


Adding 2026 (Jan–Sep), summed from the monthly table:

In [10]:
m26 = monthly[(monthly["year"] == 2026) & monthly["reported"]]
annual.loc[2026, ["accidents", "deaths", "injuries"]] = m26[["accidents", "deaths", "injuries"]].sum().values
annual.loc[2026, "coverage"] = f"partial: {len(m26)} of 9 months (Jan-Sep)"
annual.loc[2026, "source_url"] = "sum of monthly_national.csv"
annual["source"] = "RSF"
annual["deaths_per_accident"] = (annual["deaths"] / annual["accidents"]).round(2)
annual["motorcycle_death_share_pct"] = (100 * annual["motorcycle_deaths"] / annual["deaths"]).round(1)
annual["pedestrian_death_share_pct"] = (100 * annual["pedestrian_deaths"] / annual["deaths"]).round(1)
annual = annual.reset_index().rename(columns={"index": "year"})
annual.to_csv(OUT / "annual_national.csv", index=False)
annual

field,year,accidents,deaths,injuries,motorcycle_accidents,motorcycle_deaths,pedestrian_deaths,women_deaths,children_deaths,source_url,coverage,source,deaths_per_accident,motorcycle_death_share_pct,pedestrian_death_share_pct
0,2021,5371.0,6284.0,7468.0,2078.0,2214.0,1523.0,927.0,734.0,https://unb.com.bd/category/Bangladesh/perilou...,full year,RSF,1.17,35.2,24.2
1,2022,6829.0,7713.0,12615.0,2973.0,3091.0,1627.0,1061.0,1143.0,https://old.thefinancialexpress.com.bd/nationa...,full year,RSF,1.13,40.1,21.1
2,2023,6911.0,6524.0,11407.0,2532.0,2487.0,NaN,974.0,1128.0,https://www.dhakatribune.com/bangladesh/337884...,full year,RSF,0.94,38.1,NaN
3,2024,6927.0,7294.0,12019.0,2761.0,2609.0,1535.0,893.0,1152.0,https://en.prothomalo.com/bangladesh/m4kzejfpvx,full year,RSF,1.05,35.8,21.0
4,2025,7584.0,7359.0,16476.0,3029.0,2671.0,1564.0,962.0,1008.0,https://en.prothomalo.com/bangladesh/accident/...,full year,RSF,0.97,36.3,21.3
5,2026,3483.0,3181.0,6652.0,NaN,NaN,NaN,NaN,NaN,sum of monthly_national.csv,partial: 7 of 9 months (Jan-Sep),RSF,0.91,NaN,NaN


### 4.1 Cross-check: annual report vs sum of monthly reports

No year has all 12 months, so I scale the average reported month up to 12 months and compare that with the annual
report. If the two are close, the monthly figures are consistent with the annual totals. RSF revises its numbers
in the annual report, so a small gap is expected.

In [11]:
check = (
    monthly[monthly["year"] <= 2025]
    .groupby("year")
    .agg(months_reported=("reported", "sum"), mean_monthly_deaths=("deaths", "mean"))
    .join(annual.set_index("year")["deaths"].rename("annual_report"))
)
check["annualised_from_monthly"] = (12 * check["mean_monthly_deaths"]).round()
check["gap_pct"] = (100 * (check["annualised_from_monthly"] / check["annual_report"] - 1)).round(1)
check

,months_reported,mean_monthly_deaths,annual_report,annualised_from_monthly,gap_pct
year,,,,,
2021,8,455.875,6284.0,5470.0,-13.0
2022,8,573.125,7713.0,6878.0,-10.8
2023,10,472.000,6524.0,5664.0,-13.2
2024,7,546.000,7294.0,6552.0,-10.2
2025,11,524.000,7359.0,6288.0,-14.6


The monthly reports come out **11–15% below** the annual report every year, and the gap is about the same in each
year. RSF's annual report adds accidents reported late and injured people who died later. So:

- use **annual figures** for year-on-year totals;
- use **monthly figures** only to compare months and seasons with each other, never added up as a yearly total.

## 5. Deaths by vehicle type

RSF groups deaths by the vehicle the victim was in or hit by: motorcycle, bus, truck/pickup/goods vehicle,
car/microbus, three-wheeler (CNG, easy-bike, auto-rickshaw), local vehicles (nosimon, korimon, votvoti) and
bicycle/rickshaw. The full split is printed for only some months. Motorcycle deaths are reported almost every month.

In [12]:
VEHICLE_LABELS = {
    "motorcycle": "Motorcycle",
    "three_wheeler": "Three-wheeler",
    "truck_pickup_goods": "Truck / pickup / goods",
    "bus": "Bus",
    "car_microbus": "Car / microbus",
    "local_vehicle": "Local vehicle",
    "bicycle_rickshaw": "Bicycle / rickshaw",
}


def period_parts(period):
    """'2025-07' -> (2025, 7, 'month'); 'annual' handled by caller; anything else -> None."""
    try:
        p = pd.Period(period, "M")
        return p.year, p.month, "month"
    except (ValueError, TypeError):
        return None


def tidy_breakdown(dimension, measures):
    d = breakdowns[(breakdowns["source"] == "RSF") & (breakdowns["dimension"] == dimension) & breakdowns["measure"].isin(measures)].copy()
    rows = []
    for _, r in d.iterrows():
        if r["period"] == "annual":
            parts = (int(r["year"]), np.nan, "year")
        else:
            parts = period_parts(r["period"])
        if parts is None:
            continue
        rows.append({"year": parts[0], "month": parts[1], "period_type": parts[2], "category": r["category"],
                     "measure": r["measure"], "value": r["value"], "source": "RSF", "source_url": r["citation_url"],
                     "note": r["note"]})
    out = pd.DataFrame(rows)
    out = out[out["year"].between(2021, 2026)]
    return out.drop_duplicates(["year", "month", "period_type", "category", "measure"], keep="first")


veh = tidy_breakdown("vehicle_type", ["deaths"])

manual_veh = manual[manual["table"] == "vehicle"].assign(period_type="month", measure="deaths", value=lambda d: d["deaths"])
veh = pd.concat([veh, manual_veh[veh.columns]], ignore_index=True)
veh = veh.drop_duplicates(["year", "month", "period_type", "category", "measure"], keep="first")
veh = veh[veh["category"].isin(VEHICLE_LABELS)]
veh["vehicle_type"] = veh["category"].map(VEHICLE_LABELS)

# Make sure monthly rows fall inside the study window.
in_window = (veh["period_type"] == "year") | (
    pd.PeriodIndex.from_fields(year=veh["year"], month=veh["month"].fillna(1).astype(int), freq="M") <= END
)
veh = veh[in_window]

vehicle = veh[["year", "month", "period_type", "vehicle_type", "value", "source", "source_url", "note"]].rename(columns={"value": "deaths"})
vehicle = vehicle.sort_values(["year", "period_type", "month", "deaths"], ascending=[True, False, True, False])
vehicle.to_csv(OUT / "vehicle_deaths.csv", index=False)

print("Months with full 7-category split:",
      vehicle[vehicle["period_type"] == "month"].groupby(["year", "month"])["vehicle_type"].nunique().eq(7).sum())
vehicle.pivot_table(index=["year", "period_type"], columns="vehicle_type", values="deaths", aggfunc="sum")

Months with full 7-category split: 15


vehicle_type      Bicycle / rickshaw    Bus  Car / microbus  Local vehicle  Motorcycle  Three-wheeler  Truck / pickup / goods
year period_type                                                                                                             
2021 month                       NaN    NaN             NaN            NaN      1045.0           57.0                    38.0
     year                        NaN    NaN             NaN            NaN      2214.0            NaN                     NaN
2022 month                       NaN    NaN             NaN            NaN      1730.0            NaN                     NaN
     year                        NaN    NaN             NaN            NaN      3091.0            NaN                     NaN
2023 month                      39.0   30.0            31.0           29.0      1342.0          242.0                    71.0
     year                        NaN    NaN             NaN            NaN      2487.0            NaN                     NaN
2024 month                      25.0   57.0            32.0           24.0      1223.0          197.0                    53.0
     year                      196.0  369.0           363.0          331.0      2609.0         1392.0                   499.0
2025 month                      45.0  150.0            81.0           95.0      2100.0          454.0                   185.0
     year                        NaN    NaN             NaN            NaN      2671.0            NaN                     NaN
2026 month                      48.0  162.0            86.0          103.0      1100.0          510.0                   205.0

## 6. Divisions

Bangladesh has 8 divisions. The press nearly always quotes RSF's **highest** division (Dhaka in almost every month)
and **lowest** division, but prints the full 8-division table only now and then, usually as each division's share
(%) of accidents. Both kinds are kept, with a `measure` column to tell them apart.

In [13]:
DIVISIONS = ["Dhaka", "Chattogram", "Rajshahi", "Khulna", "Barishal", "Sylhet", "Rangpur", "Mymensingh"]

div = tidy_breakdown("division", ["accidents", "deaths", "share_pct"])
manual_div = manual[manual["table"] == "division"].melt(
    id_vars=["year", "month", "category", "source", "source_url", "note"], value_vars=["accidents", "deaths"],
    var_name="measure", value_name="value",
).dropna(subset=["value"]).assign(period_type="month")
div = pd.concat([div, manual_div[div.columns]], ignore_index=True)
div = div.drop_duplicates(["year", "month", "period_type", "category", "measure"], keep="first")

print("Division names found:", sorted(div["category"].unique()))
assert set(div["category"]) <= set(DIVISIONS)

division = div.rename(columns={"category": "division"})[
    ["year", "month", "period_type", "division", "measure", "value", "source", "source_url", "note"]
].sort_values(["year", "period_type", "month", "measure", "value"], ascending=[True, False, True, True, False])
division.to_csv(OUT / "division.csv", index=False)

full_tables = (
    division[(division["measure"] == "share_pct") & (division["period_type"] == "month")]
    .groupby(["year", "month"])["division"].nunique()
)
print("Months with a full 8-division share table:", full_tables[full_tables == 8].index.tolist())

Division names found: ['Barishal', 'Chattogram', 'Dhaka', 'Khulna', 'Mymensingh', 'Rajshahi', 'Rangpur', 'Sylhet']
Months with a full 8-division share table: [(2022, 6.0), (2022, 7.0), (2022, 8.0), (2025, 8.0), (2025, 9.0), (2025, 10.0), (2025, 11.0), (2026, 6.0)]


## 7. Time of day and Eid travel windows

Some RSF reports give the share of accidents by time of day. RSF also publishes separate counts for the Eid-ul-Fitr
and Eid-ul-Azha travel periods, when millions of people travel home at once.

In [14]:
tod = tidy_breakdown("time_of_day", ["share_pct"]).rename(columns={"category": "time_band", "value": "share_pct"})
tod = tod[tod["period_type"] == "month"][["year", "month", "time_band", "share_pct", "source", "source_url"]]
tod.to_csv(OUT / "time_of_day.csv", index=False)

eid = breakdowns[(breakdowns["source"] == "RSF") & breakdowns["period"].str.startswith("eid_") & (breakdowns["dimension"] == "total")]
eid = eid.drop_duplicates(["period", "measure"], keep="first")
eid_periods = eid.pivot(index="period", columns="measure", values="value")
eid_periods["source_url"] = eid.drop_duplicates("period").set_index("period")["citation_url"]
eid_periods = eid_periods.reset_index()
eid_periods["festival"] = eid_periods["period"].str.extract(r"eid_(fitr|azha)")[0].map({"fitr": "Eid-ul-Fitr", "azha": "Eid-ul-Azha"})
eid_periods["year"] = eid_periods["period"].str[-4:].astype(int)
eid_periods["source"] = "RSF"
eid_periods = eid_periods[eid_periods["year"] >= 2021][
    ["year", "festival", "days", "accidents", "deaths", "injuries", "deaths_per_day", "source", "source_url"]
].sort_values(["year", "festival"])
eid_periods.to_csv(OUT / "eid_periods.csv", index=False)
eid_periods

measure,year,festival,days,accidents,deaths,injuries,deaths_per_day,source,source_url
1,2021,Eid-ul-Azha,11.0,158.0,207.0,289.0,18.81,RSF,https://thefinancialexpress.com.bd/public/nati...
2,2022,Eid-ul-Azha,NaN,NaN,274.0,NaN,NaN,RSF,https://www.dhakatribune.com/bangladesh/384402...
3,2023,Eid-ul-Azha,NaN,303.0,324.0,NaN,NaN,RSF,https://www.dhakatribune.com/bangladesh/384402...
7,2023,Eid-ul-Fitr,14.0,240.0,285.0,454.0,20.35,RSF,https://www.dhakatribune.com/bangladesh/378096...
4,2024,Eid-ul-Azha,NaN,251.0,262.0,NaN,NaN,RSF,https://www.dhakatribune.com/bangladesh/384402...
8,2024,Eid-ul-Fitr,15.0,358.0,367.0,1500.0,24.46,RSF,https://www.dhakatribune.com/bangladesh/378096...
5,2025,Eid-ul-Azha,12.0,347.0,312.0,1057.0,26.00,RSF,https://www.dhakatribune.com/bangladesh/384402...
9,2025,Eid-ul-Fitr,11.0,257.0,249.0,553.0,22.63,RSF,https://www.dhakatribune.com/bangladesh/378096...
6,2026,Eid-ul-Azha,13.0,292.0,281.0,837.0,21.61,RSF,https://www.dhakatribune.com/bangladesh/411719...


## 8. Second source: BRTA official monthly reports

Since January 2023 the **Bangladesh Road Transport Authority (BRTA)** has published an official report every month,
compiled by its divisional offices. Each report lists accidents, deaths and injuries for all 8 divisions and deaths
for 13 vehicle types. The reports are scanned PDFs, so page 1 of each was transcribed by hand into
`data/raw/brta_monthly_reports.csv` and `data/raw/brta_vehicle_deaths.csv` (see `data/raw/brta/README.md`).

BRTA and RSF count differently, so they are kept as **separate series**, each with its own `source` column, and
never added together.

### 8.1 Checks

Every report contains its own totals, which makes the transcription checkable: the 8 division rows must add up to
the printed total, and the vehicle deaths must add up to the printed death total.

In [15]:
brta_raw = pd.read_csv(RAW / "brta_monthly_reports.csv")
brta_veh_raw = pd.read_csv(RAW / "brta_vehicle_deaths.csv")

totals = brta_raw[brta_raw["division"] == "Total"].set_index(["year", "month"])
div_sum = brta_raw[brta_raw["division"] != "Total"].groupby(["year", "month"])[["accidents", "deaths", "injuries"]].sum()
mismatch = (div_sum != totals[["accidents", "deaths", "injuries"]]).any(axis=1)
print(f"Months: {len(totals)}   months where the divisions don't add up to the printed total: {mismatch.sum()}")

veh_check = pd.DataFrame({
    "vehicle_sum": brta_veh_raw.groupby(["year", "month"])["deaths"].sum(),
    "printed_deaths": totals["deaths"],
}).dropna()
print(f"Months with a vehicle breakdown: {len(veh_check)}")
veh_check[veh_check["vehicle_sum"] != veh_check["printed_deaths"]]

Months: 45   months where the divisions don't add up to the printed total: 0
Months with a vehicle breakdown: 43


vehicle_sum  printed_deaths
year month                             
2025 2            484.0             482
     4            479.0             480
     5            491.0             490

All division tables add up. In three reports (Feb, Apr and May 2025) the vehicle figures are off by 1–2 deaths **in
the original document**: every printed percentage matches its count, so these are BRTA's own rounding or typing
slips, not transcription errors. They are kept as printed and flagged in the `note` column.

### 8.2 Tables

In [16]:
brta_monthly = totals.reset_index()[["year", "month", "accidents", "deaths", "injuries", "source", "source_url"]]
brta_days = pd.to_datetime(dict(year=brta_monthly["year"], month=brta_monthly["month"], day=1)).dt.days_in_month
brta_monthly["deaths_per_day"] = (brta_monthly["deaths"] / brta_days).round(2)
brta_monthly.to_csv(OUT / "brta_monthly_national.csv", index=False)

brta_division = brta_raw[brta_raw["division"] != "Total"][
    ["year", "month", "division", "accidents", "deaths", "injuries", "source", "source_url"]]
brta_division.to_csv(OUT / "brta_division.csv", index=False)

# BRTA's 13 vehicle types, grouped to match RSF's categories as closely as possible.
BRTA_GROUPS = {
    "motorcycle": "Motorcycle",
    "easybike": "Three-wheeler", "battery_rickshaw": "Three-wheeler", "autorickshaw": "Three-wheeler",
    "truck": "Truck / pickup / goods", "pickup": "Truck / pickup / goods", "tractor": "Truck / pickup / goods",
    "bus": "Bus",
    "car": "Car / microbus", "microbus": "Car / microbus", "ambulance": "Car / microbus",
    "van": "Van / other", "other": "Van / other",
}
brta_vehicle = brta_veh_raw.assign(vehicle_group=brta_veh_raw["vehicle_type"].map(BRTA_GROUPS))
assert brta_vehicle["vehicle_group"].notna().all()
brta_vehicle = brta_vehicle[["year", "month", "vehicle_type", "vehicle_group", "deaths", "source", "source_url", "note"]]
brta_vehicle.to_csv(OUT / "brta_vehicle_deaths.csv", index=False)

print(f"BRTA months: {len(brta_monthly)} ({brta_monthly['year'].min()}-{brta_monthly['month'].iloc[0]:02d} to "
      f"{brta_monthly['year'].max()}-{brta_monthly['month'].iloc[-1]:02d})")
brta_monthly.groupby("year")[["accidents", "deaths", "injuries"]].sum()

BRTA months: 45 (2023-01 to 2026-09)


,accidents,deaths,injuries
year,,,
2023,5495,5024,7495
2024,5856,5480,6462
2025,5695,5490,6424
2026,4285,4157,5424


**Cross-check against BRTA's own annual figure.** The 12 transcribed months of 2023 add up to exactly 5,495
accidents, 5,024 deaths and 7,495 injuries. These are the annual totals BRTA announced for 2023
([TBS](https://www.tbsnews.net/bangladesh/5024-killed-road-accidents-2023-brta-775982)), which independently
confirms the transcription.

Unlike RSF, BRTA assigns **every** death to a vehicle, so pedestrians are counted under the vehicle that hit them.
Vehicle shares from the two sources are therefore not directly comparable.

### 8.3 Combined monthly coverage

With both sources together, the only months without any figure are the RSF gaps before 2023:

In [17]:
both = monthly[["year", "month", "reported"]].merge(
    brta_monthly[["year", "month"]].assign(brta=True), on=["year", "month"], how="left").fillna({"brta": False})
both["any_source"] = both["reported"] | both["brta"]
print(f"Months in window: {len(both)}  RSF: {both['reported'].sum()}  BRTA: {both['brta'].sum()}  "
      f"either: {both['any_source'].sum()}")
both.loc[~both["any_source"], ["year", "month"]].groupby("year")["month"].apply(list)

Months in window: 69  RSF: 51  BRTA: 45  either: 61


year
2021      [2, 7, 8, 9]
2022    [1, 2, 10, 12]
Name: month, dtype: object

## 9. Summary

Files written:

In [18]:
for f in sorted(OUT.glob("*.csv")):
    df = pd.read_csv(f)
    has_data = df["reported"] if "reported" in df else pd.Series(True, index=df.index)
    missing_src = (has_data & df["source_url"].isna()).sum()
    print(f"{f.name:<22} {len(df):>4} rows   data rows without source_url: {missing_src}")

annual_national.csv       6 rows   data rows without source_url: 0
brta_division.csv       360 rows   data rows without source_url: 0
brta_monthly_national.csv   45 rows   data rows without source_url: 0
brta_vehicle_deaths.csv  559 rows   data rows without source_url: 0
division.csv            199 rows   data rows without source_url: 0
eid_periods.csv           9 rows   data rows without source_url: 0
monthly_national.csv     69 rows   data rows without source_url: 0
time_of_day.csv          34 rows   data rows without source_url: 0
vehicle_deaths.csv      152 rows   data rows without source_url: 0


**Limits of this data**

- RSF counts accidents **reported in the media**. Police records show far fewer accidents and the WHO estimates
  far more deaths, so these figures are a floor, not a full census. BRTA's official counts are lower still.
- 18 of the 69 months have no RSF figure I could confirm. The 2022 January and February report appeared only as a
  combined two-month total (848 accidents, 1,012 deaths). From 2023 on, BRTA covers every month.
- RSF vehicle-type and full division tables exist only for the months the press reproduced them; BRTA has both
  for every month from 2023 (vehicles from March 2023).